In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

OUTPUT_DIR = "/content/drive/MyDrive/ĐATN/data/processed/embeddings/text_features_taskb"
IMG_OUTPUT_DIR = "/content/drive/MyDrive/ĐATN/data/processed/embeddings/image_features_taskb"
os.makedirs(IMG_OUTPUT_DIR, exist_ok=True)

MASTER_CSV_PATH = f"{OUTPUT_DIR}/fashioniq_taskb_master.csv"
df_clean = pd.read_csv(MASTER_CSV_PATH)

print(f"Đã đọc {len(df_clean)} dòng từ bước Text Encoder.")
df_clean.head()


Mounted at /content/drive
Đã đọc 24016 dòng từ bước Text Encoder.


,category,split,reference_image_id,target_image_id,caption_1,caption_2,modification_text,word_count
0,dress,train,B003FGW7MK,B008BHCT58,is solid black with no sleeves,is black with straps,is solid black with no sleeves and is black wi...,11
1,dress,train,B008MTHLHQ,B00BZ8GPVO,is longer,is lighter and longer,is longer and is lighter and longer,7
2,dress,train,B00EVKYJAC,B008KZR6WM,3/4 sleeve black and white dress and more top ...,has short sleeve and is black color,3/4 sleeve black and white dress and more top ...,18
3,dress,train,B003U9VL4W,B00684UYEE,is patterned and has a halter neckline,is black with floral patterns,is patterned and has a halter neckline and is ...,13
4,dress,train,B00BLY2L2E,B00EV1B9C2,is black with no sleeves,is longer,is black with no sleeves and is longer,8


In [2]:
!pip install -q kaggle

from google.colab import files
os.makedirs('/root/.kaggle', exist_ok=True)
print("Vui lòng upload file kaggle.json (bỏ qua nếu đã có sẵn trong runtime):")
uploaded = files.upload()
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

DATA_ROOT = "/content/fashioniq_data"
os.makedirs(DATA_ROOT, exist_ok=True)
!kaggle datasets download -d binhhuunguyen/fashioniq -p {DATA_ROOT} --unzip

IMAGES_DIR = f"{DATA_ROOT}/resized_images/resized_images"
assert os.path.isdir(IMAGES_DIR), f"Không tìm thấy: {IMAGES_DIR}"
print(f"Ảnh sẵn sàng tại: {IMAGES_DIR}")


Vui lòng upload file kaggle.json (bỏ qua nếu đã có sẵn trong runtime):


Saving Kaggle Settings.json to Kaggle Settings.json
mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/binhhuunguyen/fashioniq
License(s): unknown
100% 632M/632M [00:36<00:00, 18.2MB/s]

Ảnh sẵn sàng tại: /content/fashioniq_data/resized_images/resized_images


## Bước 1 — Xác định danh sách ảnh duy nhất (khử trùng lặp)

1 ảnh reference có thể xuất hiện ở nhiều dòng (nhiều modifier khác nhau cùng áp lên 1 ảnh gốc). Encode theo ID duy nhất để không tốn GPU encode trùng.


In [3]:
unique_reference_ids = df_clean['reference_image_id'].unique().tolist()
unique_target_ids = df_clean['target_image_id'].unique().tolist()

print(f"Số dòng triplet: {len(df_clean)}")
print(f"Số ảnh Reference DUY NHẤT cần encode: {len(unique_reference_ids)}")
print(f"Số ảnh Target DUY NHẤT cần encode:    {len(unique_target_ids)}")
print(f"Tỉ lệ trùng lặp Reference: {(1 - len(unique_reference_ids)/len(df_clean))*100:.1f}% ảnh được dùng lại cho nhiều dòng")


Số dòng triplet: 24016
Số ảnh Reference DUY NHẤT cần encode: 16787
Số ảnh Target DUY NHẤT cần encode:    23742
Tỉ lệ trùng lặp Reference: 30.1% ảnh được dùng lại cho nhiều dòng


## Bước 2 & 3 — Dataset/DataLoader dùng CLIPImageProcessor chuẩn

Dùng processor có sẵn của Fashion-CLIP thay vì tự viết resize/normalize tay, để đảm bảo đúng chuẩn tiền xử lý lúc pretrain. Ảnh corrupt sẽ được log lại thay vì làm crash cả pipeline.


In [4]:
!pip install -q transformers torch pillow

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPImageProcessor
from PIL import Image

model_name = "patrickjohncyh/fashion-clip"
image_processor = CLIPImageProcessor.from_pretrained(model_name)

class FashionImageDataset(Dataset):
    def __init__(self, image_ids, images_dir, processor):
        self.image_ids = image_ids
        self.images_dir = images_dir
        self.processor = processor
        self.corrupted_ids = []  # log ảnh lỗi để xử lý sau

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        path = f"{self.images_dir}/{image_id}.jpg"
        try:
            img = Image.open(path).convert("RGB")
            pixel_values = self.processor(images=img, return_tensors="pt")["pixel_values"][0]
        except Exception as e:
            self.corrupted_ids.append(image_id)
            pixel_values = torch.zeros(3, 224, 224)  # placeholder, sẽ lọc bỏ theo corrupted_ids sau
        return image_id, pixel_values

BATCH_SIZE = 64
print("Đã định nghĩa Dataset. Sẵn sàng tạo DataLoader cho Reference và Target.")


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Đã định nghĩa Dataset. Sẵn sàng tạo DataLoader cho Reference và Target.


## Bước 4a — Load 2 model wrapper khác nhau

`CLIPVisionModel` = chỉ nhánh ảnh thuần (không projection) → dùng cho Reference.
`CLIPModel` = đầy đủ, có `get_image_features()` đã qua projection về 512-dim → dùng cho Target.


In [5]:
from transformers import CLIPVisionModel, CLIPModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

vision_model = CLIPVisionModel.from_pretrained(model_name).to(device)
vision_model.eval()

clip_model = CLIPModel.from_pretrained(model_name).to(device)
clip_model.eval()

print(f"Đã load xong 2 model trên: {device}")


config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Đã load xong 2 model trên: cuda


## Bước 4b — Encode Reference Image (sequence-level, cho XAA)

In [6]:
import numpy as np
import time

ref_dataset = FashionImageDataset(unique_reference_ids, IMAGES_DIR, image_processor)
ref_loader = DataLoader(ref_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

reference_id2idx = {}
all_ref_hidden_states = []

start_time = time.time()
idx_counter = 0
with torch.no_grad():
    for batch_ids, batch_pixels in ref_loader:
        batch_pixels = batch_pixels.to(device)
        outputs = vision_model(pixel_values=batch_pixels)
        hidden = outputs.last_hidden_state.cpu().numpy()  # [batch, 50, 768]

        for img_id, h in zip(batch_ids, hidden):
            reference_id2idx[img_id] = idx_counter
            all_ref_hidden_states.append(h)
            idx_counter += 1

reference_hidden_states = np.stack(all_ref_hidden_states, axis=0)
print(f"Hoàn tất Reference trong {time.time()-start_time:.2f}s")
print(f"reference_hidden_states shape: {reference_hidden_states.shape}")  # [N_ref, 50, 768]
print(f"Số ảnh lỗi (corrupt) phát hiện: {len(ref_dataset.corrupted_ids)}")
if ref_dataset.corrupted_ids:
    print(f"  ID ảnh lỗi: {ref_dataset.corrupted_ids[:10]}...")


Hoàn tất Reference trong 85.49s
reference_hidden_states shape: (16787, 50, 768)
Số ảnh lỗi (corrupt) phát hiện: 0


## Bước 4c — Encode Target Image (pooled + projected, cho Contrastive Loss & FAISS)

In [8]:
tgt_dataset = FashionImageDataset(unique_target_ids, IMAGES_DIR, image_processor)
tgt_loader = DataLoader(tgt_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

target_id2idx = {}
all_tgt_embeds = []

start_time = time.time()
idx_counter = 0
with torch.no_grad():
    for batch_ids, batch_pixels in tgt_loader:
        batch_pixels = batch_pixels.to(device)
        raw_output = clip_model.get_image_features(pixel_values=batch_pixels)

        # Tương thích 2 kiểu trả về tuỳ phiên bản transformers:
        # - Bản cũ: trả thẳng tensor [batch, 512]
        # - Bản mới (sau PR #42564): trả object BaseModelOutputWithPooling, vector nằm ở .pooler_output
        if hasattr(raw_output, "pooler_output"):
            image_embeds = raw_output.pooler_output
        else:
            image_embeds = raw_output

        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)    # L2 normalize
        embeds_np = image_embeds.cpu().numpy()

        for img_id, e in zip(batch_ids, embeds_np):
            target_id2idx[img_id] = idx_counter
            all_tgt_embeds.append(e)
            idx_counter += 1

target_embeds = np.stack(all_tgt_embeds, axis=0)
print(f"Hoàn tất Target trong {time.time()-start_time:.2f}s")
print(f"target_embeds shape: {target_embeds.shape}")  # [N_target, 512]
print(f"Số ảnh lỗi (corrupt) phát hiện: {len(tgt_dataset.corrupted_ids)}")
if tgt_dataset.corrupted_ids:
    print(f"  ID ảnh lỗi: {tgt_dataset.corrupted_ids[:10]}...")


Hoàn tất Target trong 95.77s
target_embeds shape: (23742, 512)
Số ảnh lỗi (corrupt) phát hiện: 0


## Bước 5 — Lưu kết quả theo mapping ID → index

Lưu file `.npy` + file mapping `.json` riêng — khác cách lưu theo thứ tự hàng CSV như ở Text Encoder, vì ảnh có trùng lặp giữa các dòng.


In [9]:
import json

np.save(f"{IMG_OUTPUT_DIR}/reference_image_hidden_states.npy", reference_hidden_states)
with open(f"{IMG_OUTPUT_DIR}/reference_id2idx.json", "w") as f:
    json.dump(reference_id2idx, f)

np.save(f"{IMG_OUTPUT_DIR}/target_image_embeds.npy", target_embeds)
with open(f"{IMG_OUTPUT_DIR}/target_id2idx.json", "w") as f:
    json.dump(target_id2idx, f)

print("Đã lưu toàn bộ output vào:", IMG_OUTPUT_DIR)
for fname in os.listdir(IMG_OUTPUT_DIR):
    print(" -", fname)


Đã lưu toàn bộ output vào: /content/drive/MyDrive/ĐATN/data/processed/embeddings/image_features_taskb
 - reference_image_hidden_states.npy
 - reference_id2idx.json
 - target_image_embeds.npy
 - target_id2idx.json


## Bước 6 — Sanity check & lọc lại các dòng có ảnh lỗi (nếu có)

In [10]:
# 6.1: Kiểm tra số lượng khớp giữa .npy và id2idx
assert len(reference_id2idx) == reference_hidden_states.shape[0], "LỖI: reference id2idx và .npy không khớp số lượng!"
assert len(target_id2idx) == target_embeds.shape[0], "LỖI: target id2idx và .npy không khớp số lượng!"
print("Số lượng khớp giữa id2idx và .npy cho cả Reference lẫn Target.")

# 6.2: Kiểm tra mọi ID trong df_clean đều có mặt trong id2idx
missing_ref = set(df_clean['reference_image_id']) - set(reference_id2idx.keys())
missing_tgt = set(df_clean['target_image_id']) - set(target_id2idx.keys())
print(f"Reference ID bị thiếu (do ảnh lỗi): {len(missing_ref)}")
print(f"Target ID bị thiếu (do ảnh lỗi):    {len(missing_tgt)}")

# 6.3: Nếu có thiếu, lọc lại df_clean và lưu đè master CSV để đồng bộ với Text Encoder
if missing_ref or missing_tgt:
    before = len(df_clean)
    df_clean = df_clean[
        ~df_clean['reference_image_id'].isin(missing_ref)
        & ~df_clean['target_image_id'].isin(missing_tgt)
    ].reset_index(drop=True)
    print(f"Đã loại {before - len(df_clean)} dòng do ảnh lỗi. Còn lại: {len(df_clean)} dòng.")
    df_clean.to_csv(MASTER_CSV_PATH, index=False)
    print("Đã cập nhật lại master CSV — cần chạy lại bước Text Encoder để đồng bộ lại 2 phía nếu số dòng thay đổi!")
else:
    print("Không có ảnh lỗi — không cần lọc lại.")

# 6.4: Xem thử 1 mẫu
sample_ref_id = unique_reference_ids[0]
sample_idx = reference_id2idx[sample_ref_id]
print(f"\nMẫu kiểm tra — Reference ID: {sample_ref_id}")
print(f"Hidden state shape: {reference_hidden_states[sample_idx].shape}")  # (50, 768)


Số lượng khớp giữa id2idx và .npy cho cả Reference lẫn Target.
Reference ID bị thiếu (do ảnh lỗi): 0
Target ID bị thiếu (do ảnh lỗi):    0
Không có ảnh lỗi — không cần lọc lại.

Mẫu kiểm tra — Reference ID: B003FGW7MK
Hidden state shape: (50, 768)
